# [0.6] How to Know When an Interpretability Result Is Fake - Exercises

Build compact diagnostics for fake interpretability results: label leakage, cherry-picked examples, probe overfitting, and weak steering directions.

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter0_fundamentals"
section = "part6_fake_interpretability_results"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_fake_interpretability_results.tests as tests

GT_TIER = "GT-0"
EXERCISE_ID = "0.6.fake_interpretability_results"
EXPECTED_RUNTIME = "20-30 minutes for exercises; seconds for the CUDA diagnostic preflight"
REQUIRES_GPU = False

In [ ]:
@dataclass(frozen=True)
class LabelLeakageReport:
    leaked_feature_index: int
    leaked_feature_accuracy: float
    shifted_no_leak_accuracy: float
    accuracy_gap: float
    detects_leakage: bool


@dataclass(frozen=True)
class CherryPickReport:
    selected_mean_effect: float
    population_mean_effect: float
    population_median_effect: float
    inflation_ratio: float
    detects_cherry_picking: bool


@dataclass(frozen=True)
class ProbeOverfitReport:
    train_accuracy: float
    heldout_accuracy: float
    generalization_gap: float
    detects_overfit: bool


@dataclass(frozen=True)
class FakeRandomDirectionControlReport:
    claimed_effect: float
    random_p95_effect: float
    effect_gap: float
    passes_random_control: bool
    detects_random_direction_failure: bool


@dataclass(frozen=True)
class FakeResultAuditReport:
    leakage_detected: bool
    cherry_pick_detected: bool
    probe_overfit_detected: bool
    random_direction_failure_detected: bool
    all_bogus_results_flagged: bool

## Label Leakage

Use a threshold of zero for signed probe scores, then compare a perfect leaked feature against a shifted no-leak control.

In [ ]:
def binary_accuracy(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


tests.test_binary_accuracy_thresholds_signed_scores(binary_accuracy)

In [ ]:
def label_leakage_report() -> LabelLeakageReport:
    raise NotImplementedError()


tests.test_label_leakage_report_flags_direct_label_feature(label_leakage_report)

## Cherry-Picked Evidence

Selected examples should be compared to the full population, not just presented as anecdotes.

In [ ]:
def cherry_pick_report() -> CherryPickReport:
    raise NotImplementedError()


tests.test_cherry_pick_report_compares_selected_to_population(cherry_pick_report)

## Probe Overfitting

A memorizing probe is not evidence unless the held-out result survives.

In [ ]:
def probe_overfit_report() -> ProbeOverfitReport:
    raise NotImplementedError()


tests.test_probe_overfit_report_requires_heldout_gap(probe_overfit_report)

## Random-Direction Controls

A steering direction should beat a random-direction control distribution by a real margin.

In [ ]:
def random_direction_control_report() -> FakeRandomDirectionControlReport:
    raise NotImplementedError()


tests.test_random_direction_control_report_rejects_weak_claim(random_direction_control_report)

## Aggregate Audit

The audit should pass only when every known bogus result is flagged.

In [ ]:
def fake_result_audit_report() -> FakeResultAuditReport:
    raise NotImplementedError()


tests.test_fake_result_audit_report_aggregates_all_failure_modes(fake_result_audit_report)

In [ ]:
def leakage_diagnostic() -> dict:
    return label_leakage_report().__dict__


def cherry_pick_diagnostic() -> dict:
    return cherry_pick_report().__dict__


def probe_overfit_diagnostic() -> dict:
    return probe_overfit_report().__dict__


def random_direction_diagnostic() -> dict:
    return random_direction_control_report().__dict__


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    audit = fake_result_audit_report().__dict__
    return {
        "leakage": leakage_diagnostic(),
        "cherry_pick": cherry_pick_diagnostic(),
        "probe_overfit": probe_overfit_diagnostic(),
        "random_direction": random_direction_diagnostic(),
        "audit": audit,
        "contract_passed": audit["all_bogus_results_flagged"],
    }


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
